# HELIX Revenue Prediction

## Notebook 08 — Feature Engineering

Objective

Transform cleaned data into high-quality machine learning features.

In [1]:
# import library
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
# load dataset
PROJECT_ROOT = Path("..")

DATA_PATH = PROJECT_ROOT / "data" / "interim" / "revenue_business_clean.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Age,Gender,City,Product_Category,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating,Year,Month,Day,Weekday,Quarter,Is_Weekend
0,40,Male,Ankara,Books,29.18,Digital Wallet,Mobile,14,9,True,13,4,2023,5,29,0,2,False
1,40,Male,Ankara,Home & Garden,506.35,Credit Card,Desktop,14,8,True,6,2,2023,10,12,3,4,False
2,40,Male,Ankara,Sports,1664.10,Credit Card,Mobile,15,10,True,9,4,2023,12,5,1,4,False
3,33,Male,Istanbul,Food,275.45,Digital Wallet,Desktop,16,13,True,4,4,2023,5,11,3,2,False
4,33,Male,Istanbul,Beauty,534.45,Credit Card,Mobile,14,7,True,6,4,2023,6,16,4,2,False


In [3]:
# overview
print(df.shape)

print(df.dtypes)

(17049, 18)
Age                           int64
Gender                       object
City                         object
Product_Category             object
Total_Amount                float64
Payment_Method               object
Device_Type                  object
Session_Duration_Minutes      int64
Pages_Viewed                  int64
Is_Returning_Customer          bool
Delivery_Time_Days            int64
Customer_Rating               int64
Year                          int64
Month                         int64
Day                           int64
Weekday                       int64
Quarter                       int64
Is_Weekend                     bool
dtype: object


In [4]:
# datetime feature
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12)

df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12)

df["Weekday_Sin"] = np.sin(2 * np.pi * df["Weekday"] / 7)

df["Weekday_Cos"] = np.cos(2 * np.pi * df["Weekday"] / 7)

In [5]:
# customer features - age group
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55+"],
)

In [ ]:
# behavior feature - pages per minute

df["Pages_Per_Minute"] = np.where(
    df["Session_Duration_Minutes"] > 0,
    df["Pages_Viewed"] / df["Session_Duration_Minutes"],
    0,
)

In [ ]:
# behavior feature - engagement score

df["Engagement_Score"] = df["Pages_Viewed"] * df["Session_Duration_Minutes"]

df["Rating_x_Returning"] = df["Customer_Rating"] * df["Is_Returning_Customer"].astype(
    int
)

df["Age_x_Rating"] = df["Age"] * df["Customer_Rating"]

df["Delivery_x_Session"] = df["Delivery_Time_Days"] * df["Session_Duration_Minutes"]

In [ ]:
# behavior feature - rating per delivery

df["Rating_per_Delivery"] = np.where(
    df["Delivery_Time_Days"] > 0,
    df["Customer_Rating"] / df["Delivery_Time_Days"],
    0,
)

In [9]:
# behavior feature - long session
median_session = df["Session_Duration_Minutes"].median()

df["Long_Session"] = df["Session_Duration_Minutes"] > median_session

In [ ]:
# encoding

categorical = [
    "Gender",
    "City",
    "Product_Category",
    "Payment_Method",
    "Device_Type",
    "Age_Group",
]

bool_cols = [
    "Is_Returning_Customer",
    "Is_Weekend",
    "Long_Session",
]

for col in bool_cols:
    df[col] = df[col].astype(int)

df["City_Frequency"] = df["City"].map(df["City"].value_counts())

df["Product_Frequency"] = df["Product_Category"].map(
    df["Product_Category"].value_counts()
)

df["Payment_Frequency"] = df["Payment_Method"].map(df["Payment_Method"].value_counts())

df["Device_Frequency"] = df["Device_Type"].map(df["Device_Type"].value_counts())

# Scaling Recommendation

RobustScaler

Reason

Several numerical variables contain outliers.

StandardScaler may be evaluated during model comparison.

In [11]:
# feature selection
drop_columns = ["Year"]

feature_df = df.drop(columns=drop_columns)

print(feature_df.columns)

Index(['Age', 'Gender', 'City', 'Product_Category', 'Total_Amount',
       'Payment_Method', 'Device_Type', 'Session_Duration_Minutes',
       'Pages_Viewed', 'Is_Returning_Customer', 'Delivery_Time_Days',
       'Customer_Rating', 'Month', 'Day', 'Weekday', 'Quarter', 'Is_Weekend',
       'Month_Sin', 'Month_Cos', 'Weekday_Sin', 'Weekday_Cos', 'Age_Group',
       'Customer_Type', 'Pages_Per_Minute', 'Engagement_Score',
       'Long_Session'],
      dtype='object')


In [12]:
# final feature
target = "Total_Amount"

features = [col for col in feature_df.columns if col != target]

print(features)

print()

print("Total Feature :", len(features))

['Age', 'Gender', 'City', 'Product_Category', 'Payment_Method', 'Device_Type', 'Session_Duration_Minutes', 'Pages_Viewed', 'Is_Returning_Customer', 'Delivery_Time_Days', 'Customer_Rating', 'Month', 'Day', 'Weekday', 'Quarter', 'Is_Weekend', 'Month_Sin', 'Month_Cos', 'Weekday_Sin', 'Weekday_Cos', 'Age_Group', 'Customer_Type', 'Pages_Per_Minute', 'Engagement_Score', 'Long_Session']

Total Feature : 25


In [ ]:
# save dataset

OUTPUT = PROJECT_ROOT / "data" / "processed" / "revenue_final_dataset_v2.csv"

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

feature_df.to_csv(
    OUTPUT,
    index=False,
)

print("=" * 80)
print("Dataset Saved Successfully")
print("=" * 80)
print(OUTPUT)

..\data\processed\revenue_final_dataset.csv


## Final Dataset

Target

Total_Amount

Machine Learning Task

Regression

Recommended Algorithms

- CatBoost
- LightGBM
- XGBoost
- Random Forest
- Extra Trees

Evaluation Metrics

- RMSE
- MAE
- R²

Cross Validation

5-Fold KFold